# Notebook 04.2 - Hybrid Category Clustering (BGE + MiniBatchKMeans + HDBSCAN Noise)

**Goal**: Combine BGE-large-en-v1.5 embeddings with MiniBatchKMeans clustering and post-hoc HDBSCAN noise tagging to discover product-category groupings while avoiding the sentiment-collapse and overfit issues seen in N04 and N04.1.

**Pipeline**: Arrow load -> Tier 1 emotional filter (40 words) -> BGE embed -> MiniBatchKMeans k-sweep -> HDBSCAN noise tagging -> UMAP 2D -> JSON exports

In [ ]:
# -- 0.1  Detect execution environment -----------------------------------------
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# -- 0.2  Mount Google Drive (Colab only) --------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Drive mount -- running locally.")

In [ ]:
# -- 0.3  Install required packages --------------------------------------------
if IN_COLAB:
    import subprocess
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "sentence-transformers", "hdbscan", "umap-learn", "bertopic",
        "datasets", "scikit-learn", "pandas", "numpy",
        "matplotlib", "seaborn", "tqdm",
    ], check=True)
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ], check=True)
    print("All packages installed.")
else:
    print("Skipping pip install -- manage dependencies locally.")

In [ ]:
# -- 0.4  Standard library and third-party imports -----------------------------
import os
import json
import html
import math
import time
import random
import re
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_from_disk, concatenate_datasets
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.stats import entropy
from tqdm.auto import tqdm
print("All imports successful.")

In [ ]:
# -- 0.5  Set random seeds for full reproducibility ----------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"Random seeds fixed to {RANDOM_SEED}.")

In [ ]:
# -- 0.6  Detect and configure compute device ----------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU model : {gpu_name}")
    print(f"GPU memory: {total_mem:.1f} GB")
else:
    print("No GPU detected -- embeddings will run on CPU.")

In [ ]:
# -- 0.7  Configure display and plotting defaults ------------------------------
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 20)
CYAN = "#0891B2"
NARANJA = "#EA580C"
RED = "#DC2626"
SLATE = "#334155"
NEO_SUCCESS = "#10B981"
SENTIMENT_COLORS = ["#DC2626", "#94A3B8", "#10B981"]
CLUSTER_COLORS = ["#0891B2", "#7C3AED", "#D97706", "#0D9488", "#4F46E5", "#BE185D"]
GRAY_NOISE = "#94A3B8"
GRID_COLOR = "#94A3B8"
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["grid.color"] = GRID_COLOR
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = "0.4"
print("Display defaults configured.")

In [ ]:
# -- 0.8  Define all project paths ---------------------------------------------
if IN_COLAB:
    BASE_DIR = "/content/drive/MyDrive/nlp-project/business-case-01"
else:
    BASE_DIR = os.path.expanduser("~/+Dev/nlp-businesscase")
DATASET_DIR = os.path.join(BASE_DIR, "data", "dataset")
OUTPUT_DIR  = os.path.join(BASE_DIR, "data")
PLOTS_DIR   = os.path.join(BASE_DIR, "data", "plots")
MODELS_DIR  = os.path.join(BASE_DIR, "data", "models")
N042_OUTPUT_DIR = os.path.join(BASE_DIR, "data", "n042_outputs")
N042_MODELS_DIR = os.path.join(N042_OUTPUT_DIR, "models")
N042_PLOTS_DIR  = os.path.join(N042_OUTPUT_DIR, "plots")

# Create subfolders if they don't exist
for d in [N042_OUTPUT_DIR, N042_MODELS_DIR, N042_PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR    : {BASE_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"PLOTS_DIR   : {PLOTS_DIR}")
print(f"MODELS_DIR  : {MODELS_DIR}")

## Section 1 - Data Loading

I load the Arrow dataset from N01 and merge it with sentiment predictions from N02/N03. True labels (from star ratings) provide 100% coverage; predicted labels enrich the analysis on test rows only.

In [ ]:
# -- 1.0  Validate upstream artifacts ------------------------------------------
for split in ["train", "validation", "test"]:
    split_path = os.path.join(DATASET_DIR, split)
    assert os.path.exists(split_path), f"N01 output missing: {split_path}"
PRED_PATH = os.path.join(OUTPUT_DIR, "predictions_distilbert.csv")
assert os.path.exists(PRED_PATH), f"predictions_distilbert.csv not found at {PRED_PATH}. Run N02 first."
print("Arrow splits and predictions_distilbert.csv verified.")

In [ ]:
# -- 1.1  Load Arrow dataset -- all splits -------------------------------------
dataset = load_from_disk(DATASET_DIR)
df_all = pd.concat(
    [
        dataset["train"].to_pandas(),
        dataset["validation"].to_pandas(),
        dataset["test"].to_pandas(),
    ],
    ignore_index=True,
)
print(f"Full dataset shape: {df_all.shape}")
print(f"Columns: {list(df_all.columns)}")
print(f"\nLabel distribution:")
print(df_all["label"].value_counts().sort_index())

In [ ]:
# -- 1.2  Load sentiment predictions from N02 ----------------------------------
df_preds = pd.read_csv(PRED_PATH, encoding="utf-8")
print(f"Loaded predictions: {len(df_preds):,} rows")
print(f"Columns: {list(df_preds.columns)}")

In [ ]:
# -- 1.3  Merge data with sentiment predictions --------------------------------
df_merged = df_all.merge(
    df_preds[["text", "predicted_label", "confidence"]],
    on="text",
    how="left",
)
unique_labels = sorted(df_merged["label"].unique())
label_names = {lbl: name for lbl, name in zip(unique_labels, ["Negative", "Neutral", "Positive"])}
df_merged["sentiment_name"] = df_merged["label"].map(label_names)
SENTIMENT_COL = "sentiment_name"
drop_pct = 100 * (len(df_all) - len(df_merged)) / len(df_all)
assert drop_pct <= 1.0, f"Merge dropped {drop_pct:.1f}% rows -- investigate."
print(f"Merged shape: {df_merged.shape}")
print(f"Missing values per column:")
print(df_merged.isnull().sum())

In [ ]:
# -- 1.4  Sampling strategy for large datasets ---------------------------------
MAX_SAMPLES = 220_000
if len(df_merged) > MAX_SAMPLES:
    print(f"Dataset has {len(df_merged):,} reviews -- sampling {MAX_SAMPLES:,}...")
    rng = np.random.default_rng(RANDOM_SEED)
    sample_indices = []
    for label_val in sorted(df_merged["label"].unique()):
        group_mask = df_merged["label"] == label_val
        group_idx = df_merged.index[group_mask]
        n_for_group = max(1, int(MAX_SAMPLES * len(group_idx) / len(df_merged)))
        n_for_group = min(len(group_idx), n_for_group)
        chosen = rng.choice(group_idx, size=n_for_group, replace=False)
        sample_indices.extend(chosen.tolist())
    df_merged = df_merged.loc[sample_indices].reset_index(drop=True)
    print(f"  -> Sampled down to {len(df_merged):,} reviews.")
else:
    print(f"Dataset has {len(df_merged):,} reviews -- no sampling needed.")
print("\nClass distribution:")
for lbl, count in df_merged["label"].value_counts().sort_index().items():
    pct = 100 * count / len(df_merged)
    print(f"  {label_names[lbl]:10s} ({lbl}): {count:>8,}  ({pct:5.1f}%)")

## Section 1.5 - Row-Level + Tier 1 Combined Filter

N04.1 used both Tier 1 (40 words) and Tier 2 (23 phrases), which caused text collapse -- 93.7% of reviews landed in one cluster. N04.2 uses a **two-stage filter**:

1. **Row-level filter**: Exclude reviews with < 20 words or purely generic content (e.g. "i like it thanks")
2. **Tier 1 word filter**: Remove 40 pure-emotion words from the surviving rows

This keeps only substantive, descriptive reviews and makes the model faster by using strictly necessary information.

In [ ]:
# -- 1.4a Row-Level + Tier 1 Combined Filter ---------------------------------
# CRITICAL: First exclude short/generic reviews, THEN apply Tier 1 word filter.
# This keeps only substantive, descriptive content.
if "df_merged" not in dir():
    raise NameError("Run Section 1 first (cell-load-arrow through cell-sample).")

MIN_TEXT_WORDS = 20
print("Applying combined row + word filter ...")
print(f"  MIN_TEXT_WORDS: {MIN_TEXT_WORDS}")
start = time.perf_counter()

# Step A: Row-level filter (exclude short/generic reviews)
n_before = len(df_merged)
df_merged["word_count"] = df_merged["text"].str.split().str.len()

# Criterion 1: at least MIN_TEXT_WORDS words
df_merged = df_merged[df_merged["word_count"] >= MIN_TEXT_WORDS].copy()

# Criterion 2: at least 2 distinct non-stopword tokens (exclude "i like it thanks")
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
sklearn_stop = set(ENGLISH_STOP_WORDS)

def has_min_content(text):
    tokens = [t.lower() for t in text.split() if t.isalpha()]
    non_stop = [t for t in tokens if t not in sklearn_stop]
    return len(non_stop) >= 2

n_after_len = len(df_merged)
df_merged = df_merged[df_merged["text"].apply(has_min_content)].copy()
n_after_row = len(df_merged)
n_removed_row = n_before - n_after_row
pct_removed_row = 100 * n_removed_row / n_before
print(f"\n  Rows before filtering : {n_before:,}")
print(f"  Rows removed (short)  : {n_removed_row:,} ({pct_removed_row:.1f}%)")
print(f"  Rows after row filter : {n_after_row:,}")

# Step B: Tier 1 word filter (40 words only, NO Tier 2)
STOPWORDS_TIER1 = [
    "amazing", "awesome", "awful", "beautiful", "best", "brilliant",
    "crap", "delighted", "disappointed", "dreadful", "excellent",
    "fantastic", "favorite", "garbage", "glad", "happy", "hate",
    "horrible", "incredible", "love", "loved", "lousy", "magnificent",
    "mediocre", "nice", "outstanding", "perfect", "pleased", "rubbish",
    "satisfied", "superb", "terrible", "terrific", "thrilled", "useless",
    "waste", "wonderful", "worst", "worth",
]
print(f"\n  Tier 1 stopwords: {len(STOPWORDS_TIER1)} words (NO Tier 2)")

def filter_emotional_lexicon(text: str) -> str:
    filtered = text
    for word in STOPWORDS_TIER1:
        pattern = r'\b' + re.escape(word) + r'\b'
        filtered = re.sub(pattern, '', filtered, flags=re.IGNORECASE)
    filtered = re.sub(r'\s+', ' ', filtered).strip()
    return filtered

df_merged["text_filtered"] = df_merged["text"].apply(filter_emotional_lexicon)
n_changed = (df_merged["text"] != df_merged["text_filtered"]).sum()
pct_changed = 100 * n_changed / len(df_merged)
elapsed = time.perf_counter() - start

print(f"\n  Rows after word filter: {len(df_merged):,}")
print(f"  Rows modified by Tier1: {n_changed:,} ({pct_changed:.1f}%)")
print(f"  Time                  : {elapsed:.1f}s")

# Verify with random sample (never .head() per AGENTS.md trap #4)
print("\n  Before/After examples (random sample):")
sample = df_merged.sample(n=3, random_state=RANDOM_SEED)
for i, (_, row) in enumerate(sample.iterrows(), 1):
    orig = str(row["text"])
    filt = str(row["text_filtered"])
    print(f"\n  [{i}] Original : {orig[:200]}{'...' if len(orig) > 200 else ''}")
    print(f"      Filtered : {filt[:200]}{'...' if len(filt) > 200 else ''}")

# Export
FILTERED_CSV = os.path.join(N042_OUTPUT_DIR, "n042_reviews_filtered_bge_tier1.csv")
df_merged[["text", "text_filtered", "label", "rating", "category"]].to_csv(FILTERED_CSV, index=False)
file_size_mb = os.path.getsize(FILTERED_CSV) / 1e6
print(f"\n  Filtered texts saved -> {FILTERED_CSV}")
print(f"    Rows   : {len(df_merged):,}")
print(f"    Size   : {file_size_mb:.1f} MB")


In [ ]:
# -- 1.5b  Verify filter integrity -------------------------------------------
if "df_merged" not in dir() or "text_filtered" not in df_merged.columns:
    raise NameError("Run the combined filter cell first.")
print("Filter verification:")
print(f"  Rows with text_filtered: {df_merged['text_filtered'].notna().sum():,}")
print(f"  Empty filtered texts   : {(df_merged['text_filtered'].str.len() == 0).sum():,}")
print(f"  Avg word count (orig)  : {df_merged['word_count'].mean():.1f}")
print(f"  Avg word count (filt)  : {df_merged['text_filtered'].str.split().str.len().mean():.1f}")


## Section 2 - BGE-large-en-v1.5 Embeddings

I use BAAI/bge-large-en-v1.5 (1024-dim) instead of nomic-embed (768-dim). BGE-large has shown better topic/tone separation in prior experiments. The checkpoint at .npz allows resuming without re-encoding if Colab disconnects.

In [ ]:
# -- 2.1  Load BGE-large-en-v1.5 embedding model -------------------------------
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
EMBEDDING_DIM = 1024
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=str(DEVICE))
embedding_model.max_seq_length = 256
print(f"Model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  Embedding dim : {embedding_model.get_sentence_embedding_dimension()}")
print(f"  Max seq length: {embedding_model.max_seq_length}")
print(f"  Device        : {DEVICE}")

In [ ]:
# -- 2.2  Encode filtered review texts -----------------------------------------
if "df_merged" not in dir():
    raise NameError("Run Section 1 first (cell-load-arrow through cell-sample).")
review_texts = [html.unescape(t) for t in df_merged["text_filtered"].tolist()]
NPZ_PATH = os.path.join(N042_MODELS_DIR, "n042_embeddings_bge_tier1.npz")
if os.path.exists(NPZ_PATH) and os.path.getsize(NPZ_PATH) > 0:
    print(f"Checkpoint found -- loading embeddings from {NPZ_PATH}")
    npz = np.load(NPZ_PATH)
    embeddings = npz["embeddings"]
    print(f"  Loaded shape: {embeddings.shape}")
else:
    print(f"Encoding {len(review_texts):,} reviews (batch_size=32, normalized)...")
    start = time.perf_counter()
    embeddings = embedding_model.encode(
        review_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    elapsed = time.perf_counter() - start
    print(f"\nEmbeddings shape : {embeddings.shape}")
    assert embeddings.shape == (len(df_merged), 1024), f"Dimension mismatch: expected ({len(df_merged)}, 1024), got {embeddings.shape}"
    print(f"  dtype          : {embeddings.dtype}")
    print(f"  Memory usage   : {embeddings.nbytes / 1e9:.2f} GB")
    print(f"  Time           : {elapsed/60:.1f} min")
    norms = np.linalg.norm(embeddings, axis=1)
    print(f"  Mean L2 norm   : {norms.mean():.4f} (expected ~ 1.0)")
    np.savez_compressed(NPZ_PATH, embeddings=embeddings)
    file_size_mb = os.path.getsize(NPZ_PATH) / 1e6
    print(f"\nEmbeddings saved -> {NPZ_PATH}")
    print(f"  File size: {file_size_mb:.1f} MB")

In [ ]:
# -- 2.3  Verify embedding quality -- cosine similarity check ------------------
sampled = df_merged.sample(n=5, random_state=RANDOM_SEED).reset_index(drop=True)
sample_emb = embeddings[sampled.index]
sim_matrix = np.dot(sample_emb, sample_emb.T)
print("Cosine similarity matrix (sampled reviews):")
print(sim_matrix.round(4))
print()
same_class_scores = []
diff_class_scores = []
for i in range(len(sampled)):
    for j in range(i + 1, len(sampled)):
        score = sim_matrix[i, j]
        if sampled.loc[i, "label"] == sampled.loc[j, "label"]:
            same_class_scores.append(score)
        else:
            diff_class_scores.append(score)
if same_class_scores and diff_class_scores:
    same_mean = np.mean(same_class_scores)
    diff_mean = np.mean(diff_class_scores)
    margin = same_mean - diff_mean
    print(f"Same-class pairs     : {len(same_class_scores)}  |  mean similarity: {same_mean:.4f}")
    print(f"Different-class pairs: {len(diff_class_scores)}  |  mean similarity: {diff_mean:.4f}")
    print(f"Margin (same - diff) : {margin:.4f}")
    if margin >= 0.05:
        print("Embeddings pass sanity check -- similar-class reviews are closer.")
    else:
        print("Warning: Margin < 0.05 -- embeddings may not separate classes well.")
else:
    print("Warning: Not enough same-class or different-class pairs in sample.")

## Section 3 - MiniBatchKMeans Clustering

N04 already proved MiniBatchKMeans retains ~95% of K-Means quality at O(batch*k*d) memory. Re-running K-Means wastes ~20min with no new insight, so N04.2 uses MiniBatchKMeans exclusively. I run a k-sweep (k=2...10), select the best silhouette in range 4-6, then fit with per-step tracking.

In [ ]:
# -- 3.1  K-sweep: MiniBatchKMeans only ----------------------------------------
K_VALUES = range(2, 11)
BATCH_SIZE = 1024
N_INIT = 10
mb_inertias = []
mb_silhouettes = []
mb_times = []
SIL_SAMPLE_SIZE = min(10_000, len(embeddings))
rng = np.random.default_rng(RANDOM_SEED)
sil_sample_indices = rng.choice(len(embeddings), size=SIL_SAMPLE_SIZE, replace=False)
sil_sample = embeddings[sil_sample_indices]
print(f"Sweeping MiniBatchKMeans for k = {min(K_VALUES)} ... {max(K_VALUES)}")
print(f"  Embeddings       : {len(embeddings):,} x {embeddings.shape[1]}-dim")
print(f"  Silhouette sample: {SIL_SAMPLE_SIZE:,} points")
print(f"  n_init           : {N_INIT}")
print()
for k in tqdm(K_VALUES, desc="Sweeping k"):
    t0 = time.perf_counter()
    mb = MiniBatchKMeans(
        n_clusters=k,
        batch_size=BATCH_SIZE,
        n_init=N_INIT,
        random_state=RANDOM_SEED,
        max_iter=300,
    )
    mb.fit(embeddings)
    mb_time = time.perf_counter() - t0
    mb_inertias.append(mb.inertia_)
    mb_sil = silhouette_score(sil_sample, mb.predict(sil_sample))
    mb_silhouettes.append(mb_sil)
    mb_times.append(mb_time)
    print(f"  k={k:2d}  |  Inertia={mb_inertias[-1]:>12,.0f}  Sil={mb_sil:.4f}  Time={mb_time:.1f}s")
print("\nK-sweep complete.")

In [ ]:
# -- 3.2  Plot elbow + silhouette ----------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_VALUES, mb_inertias, marker="s", color=CYAN, linewidth=2, markersize=8, label="MiniBatchKMeans")
ax1.set_title("Elbow Method -- Inertia vs. k", fontsize=13, fontweight="bold")
ax1.set_xlabel("Number of Clusters (k)", fontsize=11)
ax1.set_ylabel("Inertia (within-cluster SSE)", fontsize=11)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.axvspan(4, 6, alpha=0.1, color=SLATE, label="Target range (4-6)")
ax1.legend(fontsize=9)
ax2.plot(K_VALUES, mb_silhouettes, marker="s", color=CYAN, linewidth=2, markersize=8, label="MiniBatchKMeans")
ax2.set_title("Silhouette Score vs. k", fontsize=13, fontweight="bold")
ax2.set_xlabel("Number of Clusters (k)", fontsize=11)
ax2.set_ylabel("Silhouette Score", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.axvspan(4, 6, alpha=0.1, color=SLATE, label="Target range (4-6)")
ax2.legend(fontsize=9)
fig.suptitle("MiniBatchKMeans -- Optimal k Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
k_select_path = os.path.join(N042_PLOTS_DIR, "nb04_2_k_selection.png")
plt.savefig(k_select_path, dpi=150, bbox_inches="tight")
print(f"K-selection plot saved -> {k_select_path}")
plt.show()

In [ ]:
# -- 3.3  Select best k in range 4-6 -------------------------------------------
target_k_values = [k for k in K_VALUES if 4 <= k <= 6]
target_mb_sil = [mb_silhouettes[list(K_VALUES).index(k)] for k in target_k_values]
best_k_idx = np.argmax(target_mb_sil)
BEST_K = target_k_values[best_k_idx]
BEST_SIL = target_mb_sil[best_k_idx]
print(f"Best k in range 4-6 (MiniBatchKMeans): {BEST_K}  (Silhouette = {BEST_SIL:.4f})")
print(f"\nK-sweep results:")
print(f"  {'k':>3s}  |  {'Inertia':>15s}  {'Sil':>7s}  {'Time':>6s}")
print(f"  {'-'*42}")
for i, k in enumerate(K_VALUES):
    marker = " <- BEST" if k == BEST_K else ""
    print(f"  {k:3d}  |  {mb_inertias[i]:>15,.0f}  {mb_silhouettes[i]:>7.4f}  {mb_times[i]:>5.1f}s{marker}")

In [ ]:
# -- 3.4  Fit final MiniBatchKMeans with per-step tracking ---------------------
MB_N_EPOCHS = 10
MB_N_INIT = 3
MB_BATCH_FIT = 1024
EVAL_SAMPLE_SIZE = min(10_000, len(embeddings))
rng_fit = np.random.default_rng(RANDOM_SEED)
eval_indices = rng_fit.choice(len(embeddings), size=EVAL_SAMPLE_SIZE, replace=False)
eval_embeddings = embeddings[eval_indices]
fit_steps = []
epoch_steps = []
best_mb_inertia = float('inf')
best_mb_model = None
best_mb_labels = None
n_samples = len(embeddings)
n_batches_per_epoch = (n_samples + MB_BATCH_FIT - 1) // MB_BATCH_FIT
total_steps = MB_N_INIT * MB_N_EPOCHS * n_batches_per_epoch
print(f"Fitting MiniBatchKMeans with per-step tracking ...")
print(f"  k={BEST_K}, batch={MB_BATCH_FIT}, epochs={MB_N_EPOCHS}, inits={MB_N_INIT}")
print(f"  Batches/epoch: {n_batches_per_epoch}, total steps: {total_steps}")
print(f"  Eval sample  : {EVAL_SAMPLE_SIZE:,} reviews")
print()
t_mb_start = time.perf_counter()
step_counter = 0
for init in range(MB_N_INIT):
    mb = MiniBatchKMeans(
        n_clusters=BEST_K,
        batch_size=MB_BATCH_FIT,
        random_state=RANDOM_SEED + init,
        max_iter=1,
        n_init=1,
        reassignment_ratio=0.01,
    )
    for epoch in range(MB_N_EPOCHS):
        epoch_order = rng_fit.permutation(n_samples)
        for batch_num in range(n_batches_per_epoch):
            start = batch_num * MB_BATCH_FIT
            end = min(start + MB_BATCH_FIT, n_samples)
            batch_idx = epoch_order[start:end]
            X_batch = embeddings[batch_idx]
            t0 = time.perf_counter()
            mb.partial_fit(X_batch)
            elapsed_ms = (time.perf_counter() - t0) * 1000
            step_counter += 1
            fit_steps.append({
                "step": step_counter,
                "init": init + 1,
                "epoch": epoch + 1,
                "batch": batch_num + 1,
                "batch_size_actual": len(X_batch),
                "inertia": round(float(mb.inertia_), 2),
                "time_ms": round(elapsed_ms, 2),
            })
            if step_counter % 100 == 0 or step_counter == 1:
                print(f"[step {step_counter}/{total_steps}] init={init+1}/{MB_N_INIT} epoch={epoch+1}/{MB_N_EPOCHS} batch={batch_num+1}/{n_batches_per_epoch} inertia={mb.inertia_:,.0f}")
        eval_labels = mb.predict(eval_embeddings)
        eval_inertia = float(((eval_embeddings - mb.cluster_centers_[eval_labels]) ** 2).sum())
        epoch_steps.append({
            "init": init + 1,
            "epoch": epoch + 1,
            "eval_inertia": round(eval_inertia, 2),
            "train_inertia": round(float(mb.inertia_), 2),
        })
        print(f"  [init {init+1}/{MB_N_INIT}] Epoch {epoch+1:2d}/{MB_N_EPOCHS}  inertia={mb.inertia_:,.0f}  eval_inertia={eval_inertia:,.0f}")
    if mb.inertia_ < best_mb_inertia:
        best_mb_inertia = mb.inertia_
        best_mb_model = mb
        best_mb_labels = mb.predict(embeddings)
    print(f"  -> Init {init+1}/{MB_N_INIT} final inertia: {mb.inertia_:,.0f}\n")
mb_fit_time = time.perf_counter() - t_mb_start
FINAL_SIL = silhouette_score(sil_sample, best_mb_labels[sil_sample_indices])
print(f"MiniBatchKMeans training complete:")
print(f"  Silhouette : {FINAL_SIL:.4f}")
print(f"  Inertia    : {best_mb_model.inertia_:,.0f}")
print(f"  Fit time   : {mb_fit_time:.1f}s")
print(f"  Steps logged: {len(fit_steps)} batches, {len(epoch_steps)} epochs")
df_merged["cluster_kmeans"] = best_mb_labels
print(f"\nCluster labels assigned. Using MiniBatchKMeans as primary clusters.")

## Section 3.5 - HDBSCAN Post-hoc Noise Tagging

After MiniBatchKMeans assigns clusters, I run HDBSCAN on the same embeddings to tag outliers as noise (-1). This preserves the MiniBatchKMeans structure while isolating outliers. The retry ladder starts at min_cluster_size=1000; if noise exceeds 15%, I retry at 2000.

In [ ]:
# -- 3.5  HDBSCAN noise tagging ------------------------------------------------
print("Running HDBSCAN post-hoc noise tagging ...")
hdbscan_model = HDBSCAN(
    min_cluster_size=1000,
    metric="euclidean",
    min_samples=10,
    prediction_data=True,
)
hdb_labels = hdbscan_model.fit_predict(embeddings)
n_noise = (hdb_labels == -1).sum()
pct_noise = 100 * n_noise / len(embeddings)
final_min_cluster_size = 1000
print(f"  HDBSCAN(min_cluster_size=1000):")
print(f"    n_noise : {n_noise:,} ({pct_noise:.2f}%)")
if pct_noise > 15:
    print(f"\n  Noise > 15% -- retrying with min_cluster_size=2000 ...")
    hdbscan_retry = HDBSCAN(
        min_cluster_size=2000,
        metric="euclidean",
        min_samples=10,
        prediction_data=True,
    )
    hdb_labels = hdbscan_retry.fit_predict(embeddings)
    n_noise = (hdb_labels == -1).sum()
    pct_noise = 100 * n_noise / len(embeddings)
    final_min_cluster_size = 2000
    print(f"    n_noise : {n_noise:,} ({pct_noise:.2f}%)")
df_merged["cluster"] = df_merged["cluster_kmeans"].copy()
df_merged.loc[hdb_labels == -1, "cluster"] = -1
print(f"\nNoise tagging complete.")
print(f"  Algorithm          : HDBSCAN (post-hoc)")
print(f"  min_cluster_size   : {final_min_cluster_size}")
print(f"  n_noise            : {n_noise:,} ({pct_noise:.2f}%)")
print(f"  Non-noise reviews  : {len(df_merged) - n_noise:,}")
cluster_counts = df_merged["cluster"].value_counts().sort_index()
print(f"\nCluster size distribution (including noise):")
print(f"  {'Cluster':>10s}  |  {'Count':>8s}  |  {'%':>6s}")
print(f"  {'-'*34}")
for cid in sorted(cluster_counts.index):
    count = cluster_counts[cid]
    pct = 100 * count / len(df_merged)
    label = "Noise" if cid == -1 else f"Cluster {int(cid)}"
    print(f"  {label:>10s}  |  {count:>8,}  |  {pct:>5.1f}%")
fig, ax = plt.subplots(figsize=(8, 5))
n_clusters_found = len([c for c in cluster_counts.index if c != -1])
colors = CLUSTER_COLORS[:n_clusters_found]
if -1 in cluster_counts.index:
    bar_colors = colors + [GRAY_NOISE]
    bar_labels = [str(int(c)) if c != -1 else "Noise" for c in cluster_counts.index]
    bars = ax.bar(bar_labels, cluster_counts.values, color=bar_colors, edgecolor="white", linewidth=0.8)
else:
    bars = ax.bar(cluster_counts.index, cluster_counts.values, color=colors, edgecolor="white", linewidth=0.8)
ax.set_title(f"Cluster Size Distribution (k={n_clusters_found})", fontsize=13, fontweight="bold")
ax.set_xlabel("Cluster ID", fontsize=11)
ax.set_ylabel("Number of Reviews", fontsize=11)
max_height = cluster_counts.max()
ax.set_ylim(0, max_height * 1.2)
for bar, (cid, count) in zip(bars, cluster_counts.items()):
    pct = 100 * count / len(df_merged)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_height * 0.02,
        f"{count:,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plot_path = os.path.join(N042_PLOTS_DIR, "nb04_2_cluster_sizes.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"\nPlot saved -> {plot_path}")
plt.show()

## Section 4 - UMAP 2D Visualization

UMAP complexity scales with n*log(n), and >10k points makes scatter plots unreadable. I take a 5K stratified sample proportional to each cluster (including noise -1), then project to 2D for visualization.

In [ ]:
# -- 4.1  Stratified sample for UMAP visualization -----------------------------
if "df_merged" not in dir() or "cluster" not in df_merged.columns:
    raise NameError("Run Section 3 first to generate cluster labels.")
SAMPLE_SIZE = 5000
rng = np.random.default_rng(RANDOM_SEED)
sample_indices = []
for cid in sorted(df_merged["cluster"].unique()):
    mask = df_merged["cluster"] == cid
    idx = df_merged.index[mask]
    n_sample = max(1, int(SAMPLE_SIZE * len(idx) / len(df_merged)))
    n_sample = min(len(idx), n_sample)
    chosen = rng.choice(idx, size=n_sample, replace=False)
    sample_indices.extend(chosen.tolist())
df_sample = df_merged.loc[sample_indices].copy()
emb_sample = embeddings[sample_indices]
print(f"Stratified sample: {len(df_sample):,} reviews")
print(f"Cluster distribution in sample:")
print(df_sample["cluster"].value_counts().sort_index())

In [ ]:
# -- 4.2  UMAP scatter coloured by cluster -------------------------------------
from umap import UMAP
umap_2d = UMAP(
    n_neighbors=15,
    n_components=2,
    metric="cosine",
    random_state=RANDOM_SEED,
)
umap_coords = umap_2d.fit_transform(emb_sample)
df_sample["umap_x"] = umap_coords[:, 0]
df_sample["umap_y"] = umap_coords[:, 1]
fig, ax = plt.subplots(figsize=(10, 8))
cluster_ids = sorted([c for c in df_sample["cluster"].unique() if c != -1])
for i, cid in enumerate(cluster_ids):
    mask = df_sample["cluster"] == cid
    ax.scatter(
        df_sample.loc[mask, "umap_x"],
        df_sample.loc[mask, "umap_y"],
        c=[CLUSTER_COLORS[i % len(CLUSTER_COLORS)]],
        label=f"Cluster {int(cid)}",
        s=8,
        alpha=0.6,
        edgecolors="none",
    )
if -1 in df_sample["cluster"].values:
    noise_mask = df_sample["cluster"] == -1
    ax.scatter(
        df_sample.loc[noise_mask, "umap_x"],
        df_sample.loc[noise_mask, "umap_y"],
        c=[GRAY_NOISE],
        label="Noise",
        s=4,
        alpha=0.4,
        edgecolors="none",
    )
ax.set_title("UMAP Projection -- Coloured by Cluster (N04.2)", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(markerscale=3, fontsize=10, title="Cluster", title_fontsize=11, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(N042_PLOTS_DIR, "nb04_2_umap_clusters.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved -> {plot_path}")
plt.show()

In [ ]:
# -- 4.3  UMAP scatter coloured by sentiment -----------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
sentiment_labels_list = sorted(df_sample["label"].unique())
for i, lbl in enumerate(sentiment_labels_list):
    mask = df_sample["label"] == lbl
    ax.scatter(
        df_sample.loc[mask, "umap_x"],
        df_sample.loc[mask, "umap_y"],
        c=[SENTIMENT_COLORS[i % len(SENTIMENT_COLORS)]],
        label=label_names[lbl],
        s=8,
        alpha=0.5,
        edgecolors="none",
    )
ax.set_title("UMAP Projection -- Coloured by Sentiment", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(markerscale=3, fontsize=10, title="Sentiment", title_fontsize=11, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(N042_PLOTS_DIR, "nb04_2_umap_sentiment.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved -> {plot_path}")
plt.show()

## Section 5 - Cluster Analysis

I compute TF-IDF terms per cluster, category purity, sentiment entropy, and average rating. A healthy clustering has high category purity (clusters map to product categories) and high sentiment entropy (clusters are NOT sentiment-segregated).

In [ ]:
# -- 5.1  Top TF-IDF terms per cluster -----------------------------------------
print("Computing TF-IDF terms per cluster ...")
tfidf = TfidfVectorizer(
    max_features=5_000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
)
tfidf_matrix = tfidf.fit_transform(df_merged["text_filtered"])
feature_names = tfidf.get_feature_names_out()
print(f"TF-IDF vocabulary size: {len(feature_names):,} terms")
print(f"TF-IDF matrix shape   : {tfidf_matrix.shape}")
TOP_N_TERMS = 10
cluster_top_terms = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_mask = df_merged["cluster"] == cid
    cluster_indices = df_merged[cluster_mask].index
    cluster_tfidf = tfidf_matrix[cluster_indices]
    term_scores = cluster_tfidf.sum(axis=0).A1
    top_indices = np.argsort(term_scores)[::-1][:TOP_N_TERMS]
    cluster_top_terms[int(cid)] = [feature_names[i] for i in top_indices]
    n_in_cluster = cluster_mask.sum()
    print(f"\nCluster {int(cid)} ({n_in_cluster:,} reviews):")
    for rank, idx in enumerate(top_indices, 1):
        term = feature_names[idx]
        score = term_scores[idx]
        print(f"  {rank:2d}. {term:<30s}  (TF-IDF sum: {score:.2f})")

In [ ]:
# -- 5.2  Category purity per cluster ------------------------------------------
if "df_merged" not in dir() or "cluster" not in df_merged.columns:
    raise NameError("Run Section 3 first.")
cluster_purity = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    top_cats = cluster_df["category"].value_counts(normalize=True).head(3)
    purity = top_cats.sum() * 100
    cluster_purity[cid] = {
        "top_categories": top_cats.to_dict(),
        "purity_pct": purity,
    }
    print(f"Cluster {int(cid):>2.0f}: purity={purity:5.1f}%  |  top 3: {list(top_cats.index)}")
passing = sum(1 for v in cluster_purity.values() if v["purity_pct"] > 40)
print(f"\nClusters with purity > 40%: {passing}/{len(cluster_purity)}")
if passing >= 4:
    print("Category purity target met (>=4 clusters > 40%).")
else:
    print("Warning: Category purity below target.")

In [ ]:
# -- 5.1b c-TF-IDF terms per cluster (lightweight approximation) -------------
# BERTopic adds ~200MB dependency. We compute a lightweight c-TF-IDF
# by treating each cluster as a 'document' and running TF-IDF on the aggregate.
print("Computing c-TF-IDF terms per cluster ...")
cluster_docs = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_texts = df_merged[df_merged["cluster"] == cid]["text_filtered"].tolist()
    cluster_docs[int(cid)] = " ".join(cluster_texts)

ctfidf = TfidfVectorizer(
    max_features=5_000,
    stop_words="english",
    ngram_range=(1, 3),
    min_df=1,
    max_df=1.0,
)
ctfidf_matrix = ctfidf.fit_transform(list(cluster_docs.values()))
ctfidf_names = ctfidf.get_feature_names_out()

cluster_ctfidf_terms = {}
for idx, (cid, _) in enumerate(cluster_docs.items()):
    scores = ctfidf_matrix[idx].toarray()[0]
    top_idx = np.argsort(scores)[::-1][:TOP_N_TERMS]
    cluster_ctfidf_terms[cid] = [ctfidf_names[i] for i in top_idx]
    print(f"\nCluster {cid} -- c-TF-IDF top terms:")
    for rank, term_idx in enumerate(top_idx, 1):
        print(f"  {rank:2d}. {ctfidf_names[term_idx]:<35s} (score: {scores[term_idx]:.4f})")


In [ ]:
# -- 5.3  Sentiment entropy per cluster ----------------------------------------
cluster_entropy = {}
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    dist = cluster_df["label"].value_counts(normalize=True).sort_index()
    full_dist = pd.Series([dist.get(i, 0) for i in sorted(df_merged["label"].unique())])
    ent = entropy(full_dist, base=2)
    cluster_entropy[cid] = ent
    flag = "Warning" if ent < 1.0 else "OK"
    print(f"{flag} Cluster {int(cid):>2.0f}: entropy={ent:.3f}")
mean_ent = np.mean(list(cluster_entropy.values()))
print(f"\nMean sentiment entropy: {mean_ent:.3f} (target > 1.0 per cluster)")
flagged_clusters = [cid for cid, ent in cluster_entropy.items() if ent < 1.0]
if flagged_clusters:
    print(f"\nWarning: {len(flagged_clusters)} cluster(s) with low entropy: {flagged_clusters}")
    if len(flagged_clusters) >= 2:
        print("VALIDATION FAILURE: >=2 clusters flagged -- possible sentiment-driven clustering.")
else:
    print("\nAll clusters have high sentiment entropy -- not grouping by emotion.")

In [ ]:
# -- 5.4  Combined validation metrics table ------------------------------------
print(f"{'Cluster':>8s} | {'Size':>8s} | {'Purity%':>8s} | {'Entropy':>8s} | {'Top Terms':<40s} | {'Top Categories':<30s} | {'AvgRating':>9s}")
print("-" * 130)
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    size = len(cluster_df)
    purity = cluster_purity[cid]["purity_pct"]
    ent = cluster_entropy[cid]
    avg_rating = cluster_df["rating"].mean()
    top_terms = ", ".join(cluster_top_terms.get(int(cid), [])[:5])
    top_cats = cluster_purity[cid]["top_categories"]
    top_cat_str = ", ".join([f"{k}({v:.1%})" for k, v in list(top_cats.items())[:3]])
    print(f"{int(cid):8.0f} | {size:8,} | {purity:8.1f} | {ent:8.3f} | {top_terms:<40s} | {top_cat_str:<30s} | {avg_rating:9.2f}")

In [ ]:
# -- 5.5  Sentiment distribution heatmap ---------------------------------------
cluster_sentiment = pd.crosstab(
    df_merged[df_merged["cluster"] != -1]["cluster"],
    df_merged[df_merged["cluster"] != -1]["label"],
    normalize="index",
)
cluster_sentiment = cluster_sentiment.reindex(
    columns=sorted(df_merged["label"].unique()), fill_value=0
)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    cluster_sentiment,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "Proportion"},
)
ax.set_title("Sentiment Distribution per Cluster", fontsize=13, fontweight="bold")
ax.set_xlabel("Sentiment Label", fontsize=11)
ax.set_ylabel("Cluster ID", fontsize=11)
for i, label in enumerate(ax.get_xticklabels()):
    label.set_color(SENTIMENT_COLORS[i])
plt.tight_layout()
plot_path = os.path.join(N042_PLOTS_DIR, "nb04_2_sentiment_heatmap.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved -> {plot_path}")
plt.show()

In [ ]:
# -- 5.6  Category distribution heatmap (top 10 categories) --------------------
top_cats_global = df_merged["category"].value_counts().head(10).index.tolist()
cluster_category = pd.crosstab(
    df_merged[df_merged["cluster"] != -1]["cluster"],
    df_merged[df_merged["cluster"] != -1]["category"],
)
cluster_category = cluster_category[[c for c in top_cats_global if c in cluster_category.columns]]
cluster_category_norm = cluster_category.div(cluster_category.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    cluster_category_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "Proportion"},
)
ax.set_title("Category Distribution per Cluster (Top 10 Categories)", fontsize=13, fontweight="bold")
ax.set_xlabel("Amazon Category", fontsize=11)
ax.set_ylabel("Cluster ID", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plot_path = os.path.join(N042_PLOTS_DIR, "nb04_2_category_heatmap.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved -> {plot_path}")
plt.show()

### Sanity Check — ¿el embedding aprende o solo memoriza categorías?


In [ ]:
# ── Sanity Check: Metadata-only clustering ────────────────────────────
# What if we cluster WITHOUT reading the reviews — just category IDs + rating?
# This quantifies how much BGE embeddings actually LEARN vs just memorizing Amazon's labels.

if "df_merged" not in dir() or "embeddings" not in dir():
    raise NameError("Run Sections 1-2 first — df_merged and embeddings required.")
if "BEST_K" not in dir():
    raise NameError("Run Section 3 first — BEST_K required.")

from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import MiniBatchKMeans

# Build metadata-only feature matrix
meta_cats = pd.get_dummies(df_merged["category"])                    # 33 cols, one-hot
meta_feats = meta_cats.copy()
meta_feats["rating"] = df_merged["rating"].values                     # 1 col, 1-5 stars
meta_feats["label"] = df_merged["label"].values                       # 1 col, 0-2 sentiment

print(f"Metadata features: {meta_feats.shape[1]} dimensions (33 cats + rating + label)")
print(f"BGE embeddings:    {embeddings.shape[1]} dimensions")

# Fit MiniBatchKMeans on metadata ONLY
meta_kmeans = MiniBatchKMeans(
    n_clusters=BEST_K,
    batch_size=1024,
    n_init=3,
    random_state=RANDOM_SEED,
)
meta_labels = meta_kmeans.fit_predict(meta_feats.values)

# Compute category purity for metadata clusters
df_merged["meta_cluster"] = meta_labels
meta_purity = {}
for cid in range(BEST_K):
    cat_counts = df_merged[df_merged["meta_cluster"] == cid]["category"].value_counts()
    meta_purity[cid] = {
        "size": (df_merged["meta_cluster"] == cid).sum(),
        "purity_pct": 100 * cat_counts.iloc[0] / cat_counts.sum() if len(cat_counts) > 0 else 0,
        "top_category": cat_counts.index[0] if len(cat_counts) > 0 else "none",
    }

# Compare: BGE vs Metadata
print(f"\n{'='*60}")
print(f"  SANITY CHECK — BGE Embeddings vs Metadata-Only")
print(f"{'='*60}")
print(f"  {'Cluster':>8s} | {'Metadata Purity':>15s} | {'BGE Purity':>11s} | {'Winner':<12s}")
print(f"  {'-'*55}")
for cid in range(BEST_K):
    bge_purity = cluster_purity.get(cid, {}).get("purity_pct", 0) if 'cluster_purity' in dir() else 0
    meta_p = meta_purity[cid]["purity_pct"]
    bge_p = bge_purity
    winner = "BGE 🔵" if bge_p > meta_p else ("Meta ⚪" if meta_p > bge_p else "Tie")
    print(f"  {cid:8d} | {meta_p:14.1f}% | {bge_p:10.1f}% | {winner:<12s}  {meta_purity[cid]['top_category']}")

# Delta: how much does BGE improve over metadata?
meta_mean = np.mean([v["purity_pct"] for v in meta_purity.values()])
bge_mean = np.mean([cluster_purity[c]["purity_pct"] for c in cluster_purity]) if 'cluster_purity' in dir() else 0
delta = bge_mean - meta_mean
print(f"\n  Mean metadata purity : {meta_mean:.1f}%")
print(f"  Mean BGE purity      : {bge_mean:.1f}%")
print(f"  Delta (BGE − meta)   : {delta:+.1f}%")
if delta > 5:
    print(f"  ✅ BGE embeddings discover cross-category patterns (+{delta:.0f}pp)")
elif delta > -5:
    print(f"  ⚠️  BGE adds marginal value ({delta:+.0f}pp) — most signal already in categories")
else:
    print(f"  ❌ BGE HURTS purity — embeddings are confounding, not helping")
print(f"{'='*60}")


In [ ]:
# ── Export Sanity Check JSON ───────────────────────────────────────────
if "meta_purity" in dir() and "cluster_purity" in dir() and "BEST_K" in dir():
    sanity_check = {
        "comparison": "BGE Embeddings vs Metadata-Only Clustering",
        "embedding_model": EMBEDDING_MODEL_NAME,
        "k": BEST_K,
        "means": {
            "metadata_purity_mean": round(meta_mean, 1),
            "bge_purity_mean": round(bge_mean, 1),
            "delta_pp": round(delta, 1),
            "interpretation": "BGE discovers cross-category patterns" if delta > 5 else ("Marginal value" if delta > -5 else "Embeddings confound categories")
        },
        "clusters": []
    }
    for cid in range(BEST_K):
        bge_p = cluster_purity[cid]["purity_pct"] if cid in cluster_purity else 0
        sanity_check["clusters"].append({
            "id": int(cid),
            "metadata_purity_pct": round(meta_purity[cid]["purity_pct"], 1),
            "bge_purity_pct": round(bge_p, 1),
            "delta_pct": round(bge_p - meta_purity[cid]["purity_pct"], 1),
            "top_metadata_category": meta_purity[cid]["top_category"],
            "size": int(meta_purity[cid]["size"])
        })

    sanity_path = os.path.join(N042_OUTPUT_DIR, "n042_sanity_check.json")
    with open(sanity_path, "w") as f:
        json.dump(sanity_check, f, indent=2)
    print(f"\u2705 Sanity check exported \u2192 {sanity_path}")
else:
    print("\u26a0\ufe0f  Sanity check variables not found — skipping export.")


## Section 6 - Dashboard JSON Exports

I export three JSON files:
1. `n042_cluster_assignments_bge.json` -- Chart.js bubble chart datasets (5K UMAP points)
2. `n042_cluster_profiles_bge.json` -- Per-cluster profiles (terms, entropy, purity, rating)
3. `n042_training_history_bge.json` -- Full training history (fit_steps, epoch_steps, k-sweep, HDBSCAN noise)

The bubble chart format keeps the JSON at ~500KB instead of 109MB (N04.1's full row export).

In [ ]:
# -- 6.1  Export cluster assignments (bubble chart JSON) -----------------------
bubble_datasets = []
cluster_ids = sorted([c for c in df_sample["cluster"].unique() if c != -1])
for i, cid in enumerate(cluster_ids):
    mask = df_sample["cluster"] == cid
    cluster_data = df_sample[mask]
    color = CLUSTER_COLORS[i % len(CLUSTER_COLORS)]
    n_in_cluster = (df_merged["cluster"] == cid).sum()
    base_r = max(4, min(12, int(np.log1p(n_in_cluster) / 2)))
    data_points = []
    for _, row in cluster_data.iterrows():
        data_points.append({
            "x": round(float(row["umap_x"]), 2),
            "y": round(float(row["umap_y"]), 2),
            "r": base_r,
        })
    bubble_datasets.append({
        "label": f"C{int(cid)}: {cluster_top_terms.get(int(cid), ['Cluster'])[0].title()}",
        "data": data_points,
        "backgroundColor": color,
    })
if -1 in df_sample["cluster"].values:
    noise_mask = df_sample["cluster"] == -1
    noise_data = []
    for _, row in df_sample[noise_mask].iterrows():
        noise_data.append({
            "x": round(float(row["umap_x"]), 2),
            "y": round(float(row["umap_y"]), 2),
            "r": 3,
        })
    bubble_datasets.append({
        "label": "Noise",
        "data": noise_data,
        "backgroundColor": GRAY_NOISE,
    })
cluster_assignments = {
    "datasets": bubble_datasets,
    "metadata": {
        "embedding_model": EMBEDDING_MODEL_NAME,
        "clustering_algo": "MiniBatchKMeans + HDBSCAN noise",
        "k": int(BEST_K),
        "total_reviews": int(len(df_merged)),
        "umap_sample_size": int(len(df_sample)),
        "n_noise": int(n_noise),
        "silhouette": round(float(FINAL_SIL), 4),
    }
}
assign_path = os.path.join(N042_OUTPUT_DIR, "n042_cluster_assignments_bge.json")
with open(assign_path, "w", encoding="utf-8") as f:
    json.dump(cluster_assignments, f, ensure_ascii=False, indent=2)
file_size_mb = os.path.getsize(assign_path) / 1e6
print(f"n042_cluster_assignments_bge.json saved -> {assign_path}")
print(f"  Datasets : {len(bubble_datasets)}")
print(f"  Points   : {len(df_sample):,}")
print(f"  Size     : {file_size_mb:.2f} MB")
with open(assign_path, "r", encoding="utf-8") as f:
    validate = json.load(f)
assert "datasets" in validate and "metadata" in validate
assert len(validate["datasets"]) > 0
print("  Validation: JSON schema OK")

In [ ]:
# -- 6.2  Export cluster profiles ----------------------------------------------
profiles = []
for cid in sorted(df_merged["cluster"].unique()):
    if cid == -1:
        continue
    cluster_df = df_merged[df_merged["cluster"] == cid]
    size = int(len(cluster_df))
    pct = round(100 * size / len(df_merged), 2)
    top_terms = cluster_top_terms.get(int(cid), [])[:10]
    top_cats = cluster_purity[cid]["top_categories"]
    purity = round(cluster_purity[cid]["purity_pct"], 2)
    ent = round(cluster_entropy[cid], 3)
    avg_rating = round(cluster_df["rating"].mean(), 2)
    sent_dist = cluster_df["label"].value_counts(normalize=True).sort_index()
    sent_dist_pct = {str(int(k)): round(v * 100, 1) for k, v in sent_dist.items()}
    profiles.append({
        "cluster_id": int(cid),
        "size": size,
        "pct_of_total": pct,
        "top_terms": top_terms,
        "top_terms_ctfidf": top_terms[:5],
        "sentiment_distribution_pct": sent_dist_pct,
        "sentiment_entropy": ent,
        "avg_rating": avg_rating,
        "top_categories": top_cats,
        "category_purity": purity,
        "label": top_terms[0].title() if top_terms else f"Cluster {int(cid)}",
    })
profiles_path = os.path.join(N042_OUTPUT_DIR, "n042_cluster_profiles_bge.json")
with open(profiles_path, "w", encoding="utf-8") as f:
    json.dump({"clusters": profiles}, f, ensure_ascii=False, indent=2)
file_size_kb = os.path.getsize(profiles_path) / 1e3
print(f"n042_cluster_profiles_bge.json saved -> {profiles_path}")
print(f"  Clusters : {len(profiles)}")
print(f"  Size     : {file_size_kb:.1f} KB")
with open(profiles_path, "r", encoding="utf-8") as f:
    validate = json.load(f)
assert "clusters" in validate
print("  Validation: JSON schema OK")

In [ ]:
# -- 6.3  Export training history ----------------------------------------------
training_history = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dim": EMBEDDING_DIM,
    "clustering_algo": "MiniBatchKMeans + HDBSCAN",
    "best_k": int(BEST_K),
    "final_silhouette": round(float(FINAL_SIL), 4),
    "total_reviews": int(len(df_merged)),
    "training_history": {
        "minibatch_kmeans": {
            "n_inits": MB_N_INIT,
            "n_epochs": MB_N_EPOCHS,
            "batch_size": MB_BATCH_FIT,
            "total_steps": len(fit_steps),
            "fit_time_sec": round(mb_fit_time, 1),
            "fit_steps": fit_steps,
            "epoch_steps": epoch_steps,
        }
    },
    "k_sweep": {
        "k_values": list(K_VALUES),
        "minibatch_kmeans": {
            "inertias": [float(v) for v in mb_inertias],
            "silhouettes": [float(v) for v in mb_silhouettes],
            "times_sec": [round(float(v), 1) for v in mb_times],
        }
    },
    "hdbscan_noise": {
        "min_cluster_size": final_min_cluster_size,
        "noise_count": int(n_noise),
        "noise_pct": round(pct_noise, 2),
    }
}
history_path = os.path.join(N042_OUTPUT_DIR, "n042_training_history_bge.json")
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(training_history, f, ensure_ascii=False, indent=2)
file_size_kb = os.path.getsize(history_path) / 1e3
print(f"n042_training_history_bge.json saved -> {history_path}")
print(f"  Fit steps  : {len(fit_steps):,}")
print(f"  Epoch steps: {len(epoch_steps):,}")
print(f"  Size       : {file_size_kb:.1f} KB")
with open(history_path, "r", encoding="utf-8") as f:
    validate = json.load(f)
assert "training_history" in validate and "k_sweep" in validate and "hdbscan_noise" in validate
print("  Validation: JSON schema OK")

## Section 7 - Output Checklist & Final Summary

I verify that all expected output files were created with non-zero size before wrapping up.

In [ ]:
# -- 7.0  Results summary + output checklist -----------------------------------
print("=" * 60)
print("NOTEBOOK 04.2 -- HYBRID CATEGORY CLUSTERING (BGE + MiniBatchKMeans)")
print("=" * 60)
print(f"\nClustering Algorithm : MiniBatchKMeans + HDBSCAN noise")
print(f"   Best k               : {BEST_K}")
print(f"   Noise points         : {n_noise:,} ({pct_noise:.2f}%)")
print(f"   HDBSCAN min_cluster_size: {final_min_cluster_size}")
print(f"\nValidation Metrics:")
passing_purity = sum(1 for v in cluster_purity.values() if v['purity_pct'] > 40)
print(f"   Category purity >40% : {passing_purity}/{len(cluster_purity)} clusters")
mean_ent = np.mean(list(cluster_entropy.values()))
print(f"   Mean sentiment entropy: {mean_ent:.3f}")
flagged = [cid for cid, ent in cluster_entropy.items() if ent < 1.0]
print(f"   Clusters flagged (entropy<1.0): {len(flagged)}")
print(f"   Silhouette score     : {FINAL_SIL:.4f}")
print(f"\nOutput Files Checklist:")
files_to_check = [
    ("n042_reviews_filtered_bge_tier1.csv", N042_OUTPUT_DIR),
    ("n042_embeddings_bge_tier1.npz", N042_MODELS_DIR),
    ("n042_cluster_assignments_bge.json", N042_OUTPUT_DIR),
    ("n042_cluster_profiles_bge.json", N042_OUTPUT_DIR),
    ("n042_training_history_bge.json", N042_OUTPUT_DIR),
    ("n042_centroids_bge.npy", N042_MODELS_DIR),
    ("nb04_2_k_selection.png", N042_PLOTS_DIR),
    ("nb04_2_cluster_sizes.png", N042_PLOTS_DIR),
    ("nb04_2_umap_clusters.png", N042_PLOTS_DIR),
    ("nb04_2_umap_sentiment.png", N042_PLOTS_DIR),
    ("nb04_2_sentiment_heatmap.png", N042_PLOTS_DIR),
    ("nb04_2_category_heatmap.png", N042_PLOTS_DIR),
]
all_ok = True
for fname, fdir in files_to_check:
    fpath = os.path.join(fdir, fname)
    ok = os.path.exists(fpath) and os.path.getsize(fpath) > 0
    status = "OK" if ok else "MISSING"
    if not ok:
        all_ok = False
    print(f"   {status} {fname}")
if all_ok:
    print("\nN04.2 COMPLETE -- Ready for N05")
else:
    print("\nWarning: Some outputs are missing.")
print(f"\nNext Steps:")
print("   1. Web dashboard can load n042_cluster_assignments_bge.json")
print("   2. n042_cluster_profiles_bge.json feeds cluster overview cards")
print("   3. n042_training_history_bge.json enables training visualisation")

In [ ]:
# -- 6.4  Export cluster centroids for Playground matching ---------------------
if "best_mb_model" not in dir():
    raise NameError("Run Section 3 first to fit MiniBatchKMeans.")
centroids = best_mb_model.cluster_centers_
CENTROIDS_PATH = os.path.join(N042_MODELS_DIR, "n042_centroids_bge.npy")
np.save(CENTROIDS_PATH, centroids)
print(f"Cluster centroids saved -> {CENTROIDS_PATH}")
print(f"  Shape: {centroids.shape}")
print(f"  Size : {os.path.getsize(CENTROIDS_PATH) / 1e3:.1f} KB")
